# **Risk Model Regressions**

This notebook processes 43 stocks over 80 dated four-week return observations.  
\
It estimates   
(1) designated foreign-currency exposure,    
(2) AUD exposure for every stock, and   
(3) sector-factor exposure using iterative backward elimination.

In [ ]:
from pathlib import Path
import os

ROOT = Path.cwd().parent  # group4

DATA = ROOT / "data"
RAW = DATA / "raw"
PROCESSED = DATA / "processed"

OUT = ROOT / "results"
COEFF = OUT / "coefficients"
RESID = OUT / "residuals"
DIAG = OUT / "diagnostics"

import numpy as np
import pandas as pd
import statsmodels.api as sm

In [ ]:
for folder in ["raw", "processed"]:
    path = DATA / folder

    print(f"\n{folder.upper()} DATA:")

    if path.exists():
        for file in sorted(path.glob("*.xlsx")):
            print(f"  {file.name}")
    else:
        print(f"Folder not found: {path.resolve()}")

### **Data Setup**

In [ ]:
# Time-weighted 4-week returns
df_universe = pd.read_excel(
    PROCESSED / "universe_logret_4w_timeweighted_all.xlsx",
    index_col=0
)

df_stocks = pd.read_excel(
    PROCESSED / "equities_logret_4w_timeweighted_43stocks.xlsx",
    index_col=0
)

# Australian dollar 
df_aud = pd.read_excel(
    PROCESSED / "australian_dollar.xlsx",
    index_col=0
)

# Sector residual returns (market effect removed)
df_sectors = pd.read_excel(
    PROCESSED / "gics11_sector_logret_4w_spy_resid.xlsx",
    index_col=0
)

In [ ]:
datasets = {
    "Universe": df_universe,
    "Stocks": df_stocks,
    "AUD": df_aud,
    "Sectors": df_sectors
}

for name, df in datasets.items():
    print(f"\n{name}")
    print("Shape:", df.shape)
    print("Missing values:", df.isna().sum().sum())
    print("Columns:", df.columns.tolist())

## **1. FX Regressions for Foreign Stocks**

**Results:** Only NTR has a statistically significant relationship with CAD

In [ ]:

# Stock-to-currency mapping
stock_fx_map = {
    "VIST": "MXN/USD",
    "NTR": "CAD/USD",
    "ASND": "DKK/USD",
    "NOK": "EUR/USD",
    "STX": "SGD/USD"
}

In [ ]:
# Retain all 43 stocks
df_post_fx = df_stocks.copy()

fx_results = []

for stock, currency in stock_fx_map.items():

    # Align stock and currency returns by date
    data = pd.concat([
        df_stocks[stock].rename("stock"),
        df_universe[currency].rename("fx")
    ], axis=1).dropna()

    y = data["stock"]
    X = sm.add_constant(data["fx"])

    results = sm.OLS(y, X).fit()

    # Replace original returns with FX residuals
    df_post_fx[stock] = pd.NA
    df_post_fx.loc[data.index, stock] = results.resid

    fx_results.append({
        "stock": stock,
        "currency": currency,
        "alpha": results.params["const"],
        "beta": results.params["fx"],
        "std_err": results.bse["fx"],
        "t_value": results.tvalues["fx"],
        "p_value": results.pvalues["fx"],
        "r_squared": results.rsquared
    })

df_fx_results = pd.DataFrame(fx_results)

# Export FX regression coefficients
df_fx_results.to_excel(
    COEFF / "equities_fx_ols_betas.xlsx",
    index=False
)

# Export stock returns after FX adjustment
df_post_fx.to_excel(
    RESID / "equities_logret_4w_post_fx_resid.xlsx",
    index_label="date"
)

print(df_fx_results)
print("Post-FX DataFrame shape:", df_post_fx.shape)

## **2. AUD Regressions for All Stocks**

We're estimating:
$$
Y_{i,t}=\alpha_i+\beta_{i,AUD}R_{AUD,t}+\epsilon_{i,t}
$$

Where:
$$
Y_{i,t}=
\begin{cases}
\epsilon_{NTR,t}^{CAD},&i=NTR\\
R_{i,t},&i\neq NTR
\end{cases}
$$

**Results:** 27 stocks have statistically significant relationship with AUD.

In [ ]:
# Begin with original returns for all 43 stocks
df_pre_aud = df_stocks.copy()

# Only NTR retains its CAD regression residuals
df_pre_aud["NTR"] = df_post_fx["NTR"].astype(float)

In [ ]:
aud_results = []

df_post_aud = pd.DataFrame(
    index=df_pre_aud.index,
    columns=df_pre_aud.columns,
    dtype=float
)

for stock in df_pre_aud.columns:

    # Align observations by date
    data = pd.concat([
        df_pre_aud[stock].rename("stock"),
        df_aud
    ], axis=1).dropna()

    y = data["stock"]
    X = sm.add_constant(data["AUD/USD"])

    results = sm.OLS(y, X).fit()

    # Store AUD regression residuals
    df_post_aud.loc[data.index, stock] = results.resid

    # Store regression statistics
    aud_results.append({
        "stock": stock,
        "alpha": results.params["const"],
        "beta_aud": results.params["AUD/USD"],
        "std_err": results.bse["AUD/USD"],
        "t_value": results.tvalues["AUD/USD"],
        "p_value": results.pvalues["AUD/USD"],
        "r_squared": results.rsquared
    })

df_aud_results = pd.DataFrame(aud_results)

# Export AUD regression coefficients
df_aud_results.to_excel(
    COEFF / "equities_aud_ols_betas.xlsx",
    index=False
)

# Export stock returns after AUD adjustment
df_post_aud.to_excel(
    RESID / "equities_logret_4w_post_aud_resid.xlsx",
    index_label="date"
)

print(df_aud_results)
print("Post-AUD DataFrame shape:", df_post_aud.shape)

In [ ]:
# Begin with the returns before AUD regression
df_pre_sector = df_pre_aud.copy()

# Identify stocks with significant AUD exposure
significant_stocks = df_aud_results.loc[
    df_aud_results["t_value"].abs() >= 2,
    "stock"
].tolist()

# Replace only qualifying stocks with AUD residuals
df_pre_sector[significant_stocks] = df_post_aud[significant_stocks]

# Check results
print("Qualifying stocks:", len(significant_stocks))
print("Non-qualifying stocks:", 43 - len(significant_stocks))
print("Final shape:", df_pre_sector.shape)

# Export
df_pre_sector.to_excel(
    PROCESSED / "equities_logret_4w_pre_sector.xlsx",
    index_label="date"
)

## **3. Sector Multiple Regression for All Stocks**

$$
\boxed{
R_{i,t}^{\text{pre-sector}}
=
\alpha_i+
\sum_{k=1}^{11}\beta_{i,k}F_{k,t}
+\epsilon_{i,t}
}
$$

Here:
- **Y**: df_pre_sector — 43 stock return series after your currency-selection stages.
- **X**: The 11 market-adjusted sector residual-return factors.
- **Output**: Retained sector betas, t-statistics, and final sector regression residuals.  

**Sector ETFs**

- XLB - Materials
- XLC - Communications
- XLE - Energy
- XLF - Financials 
- XLI - Industrials 
- XLK - Information Technology
- XLP - Consumer Staples
- XLRE - Real Estate
- XLU - Utilities 
- XLV - Health Care
- XLY - Consumer Discretionary

In [ ]:
# Inputs: df_pre_sector and df_sectors
sector_results = []
elimination_log = []

df_post_sector = pd.DataFrame(
    index=df_pre_sector.index,
    columns=df_pre_sector.columns,
    dtype=float
)

for stock in df_pre_sector.columns:

    # Align returns and factors by date
    data = pd.concat(
        [df_pre_sector[stock].rename("stock"), df_sectors],
        axis=1
    ).dropna()

    y = data["stock"].astype(float)
    factors = df_sectors.columns.tolist()

    iteration = 0

    # Backward elimination
    while True:

        X = sm.add_constant(
            data[factors],
            has_constant="add"
        )

        model = sm.OLS(y, X).fit()

        if not factors:
            break

        t_stats = model.tvalues.drop("const")

        weakest = t_stats.abs().idxmin()
        weakest_t = t_stats[weakest]

        # Stop if all remaining factors are significant
        if abs(weakest_t) >= 2:
            break

        iteration += 1

        elimination_log.append({
            "stock": stock,
            "iteration": iteration,
            "removed_factor": weakest,
            "t_stat": weakest_t
        })

        factors.remove(weakest)

    # Save final residuals
    df_post_sector.loc[data.index, stock] = model.resid

    # Save retained factor coefficients
    for factor in factors:
        sector_results.append({
            "stock": stock,
            "factor": factor,
            "beta": model.params[factor],
            "std_err": model.bse[factor],
            "t_stat": model.tvalues[factor],
            "p_value": model.pvalues[factor],
            "r_squared": model.rsquared
        })

df_sector_results = pd.DataFrame(sector_results)
df_elimination_log = pd.DataFrame(elimination_log)

# Export
df_sector_results.to_excel(
    COEFF / "equities_sector_ols_selected_betas.xlsx",
    index=False
)

df_post_sector.to_excel(
    RESID / "equities_logret_4w_post_sector_resid.xlsx",
    index_label="date"
)

df_elimination_log.to_excel(
    DIAG / "equities_sector_elimination_log.xlsx",
    index=False
)

print("Residual matrix:", df_post_sector.shape)
print("Retained coefficients:", len(df_sector_results))
print("Factors eliminated:", len(df_elimination_log))

## **4. Market Regression for All Stocks**

$$
\boxed{
\epsilon_{i,t}^{Sector}
=
\alpha_i+\beta_{i,M}R_{SPY,t}+u_{i,t}
}
$$

Where:
- Y: Each stock's residual returns after your currency and sector regressions.
- X: SPY's 4-week log returns.
- $\beta_{i,M}$: The stock's remaining sensitivity to market movements.
- $u_{i,t}$: Residual returns unexplained by the market regression.


In [ ]:
# SPY is our market factor
spy = df_universe["SPY"].rename("market")

market_results = []
df_post_market = pd.DataFrame(
    index=df_post_sector.index,
    columns=df_post_sector.columns,
    dtype=float
)

for stock in df_post_sector.columns:

    data = pd.concat(
        [df_post_sector[stock].rename("stock"), spy],
        axis=1
    ).dropna()

    y = data["stock"]
    X = sm.add_constant(data[["market"]])

    model = sm.OLS(y, X).fit()

    market_results.append({
        "stock": stock,
        "market_beta": model.params["market"],
        "t_stat": model.tvalues["market"],
        "p_value": model.pvalues["market"],
        "r_squared": model.rsquared
    })

    df_post_market.loc[data.index, stock] = model.resid

df_market_results = pd.DataFrame(market_results)

# Export
df_market_results.to_excel(
    COEFF / "equities_market_ols_selected_betas.xlsx",
    index=False
)

df_post_market.to_excel(
    RESID / "equities_logret_4w_post_market_resid.xlsx",
    index_label="date"
)

print("Residual matrix:", df_post_market.shape)
print("Market coefficients:", len(df_market_results))

### Creating final residual matrix
For stocks with significant market exposure, keep market residuals. Else, use sector residuals.

In [ ]:
# Start with post-sector residuals for all 43 stocks
df_final_residuals = df_post_sector.copy()

# Identify stocks with significant market exposure
significant_stocks = df_market_results.loc[
    df_market_results["t_stat"].abs() > 2,
    "stock"
].tolist()

# Replace residuals only for significant stocks
df_final_residuals[significant_stocks] = df_post_market[significant_stocks]

# Check results
print("Significant market stocks:", len(significant_stocks))
print("Final residual matrix shape:", df_final_residuals.shape)

display(df_final_residuals.head())

### Constructing Covariance Matrix

In [ ]:
# Remove rows containing missing observations
df_pca_input = df_final_residuals.dropna()

# Calculate residual covariance matrix
df_residual_cov = df_pca_input.cov()

print("Observations:", len(df_pca_input))
print("Covariance matrix shape:", df_residual_cov.shape)

In [ ]:
assert np.allclose(
    df_residual_cov,
    df_residual_cov.T
)

print("Covariance matrix is symmetric.")

In [ ]:
df_residual_cov.to_excel(
    OUT / "equities_residual_cov_4w.xlsx",
    index_label="stock"
)

### Get Eigenvalues and Eigenvectors

In [ ]:
# Use complete observations across all 43 stocks
df_pca_input = df_final_residuals.astype(float).dropna()

n_obs, n_stocks = df_pca_input.shape

print(f"Observations used: {n_obs}")
print(f"Number of stocks: {n_stocks}")

assert n_stocks == 43, "Expected 43 stocks."
assert n_obs >= 2, "Insufficient observations for covariance."

In [ ]:
df_residual_cov = df_pca_input.cov()

# Verify symmetry
assert np.allclose(
    df_residual_cov.to_numpy(),
    df_residual_cov.to_numpy().T
)

print("Covariance matrix:", df_residual_cov.shape)

In [ ]:
eigenvalues, eigenvectors = np.linalg.eigh(
    df_residual_cov.to_numpy()
)

# Sort eigenvalues and their eigenvectors in descending order
idx = np.argsort(eigenvalues)[::-1]

eigenvalues = eigenvalues[idx]
eigenvectors = eigenvectors[:, idx]

# Verify nonnegative eigenvalues (allow rounding errors)
assert eigenvalues.min() >= -1e-10

In [ ]:
explained_variance = eigenvalues / eigenvalues.sum()
cumulative_variance = np.cumsum(explained_variance)

pc_names = [
    f"PC{i+1}" for i in range(len(eigenvalues))
]

df_pca_summary = pd.DataFrame({
    "PC": pc_names,
    "eigenvalue": eigenvalues,
    "explained_variance": explained_variance,
    "cumulative_variance": cumulative_variance
})

display(df_pca_summary.head(15))

# Components needed for variance thresholds
n_components_80 = np.searchsorted(
    cumulative_variance, 0.80
) + 1

n_components_90 = np.searchsorted(
    cumulative_variance, 0.90
) + 1

print("PCs for 80% variance:", n_components_80)
print("PCs for 90% variance:", n_components_90)

In [ ]:
df_eigenvectors = pd.DataFrame(
    eigenvectors,
    index=df_residual_cov.index,
    columns=pc_names
)

display(df_eigenvectors.head())

In [ ]:
# Center each stock's residual returns
E_centered = df_pca_input - df_pca_input.mean()

# Project centered residuals onto eigenvectors
df_pc_returns = E_centered @ df_eigenvectors

display(df_pc_returns.head())

In [ ]:
# Reconstruct original covariance matrix
reconstructed_cov = (
    eigenvectors
    @ np.diag(eigenvalues)
    @ eigenvectors.T
)

assert np.allclose(
    reconstructed_cov,
    df_residual_cov.to_numpy()
)

# PCs should be uncorrelated, with variances = eigenvalues
pc_cov = df_pc_returns.cov().to_numpy()

assert np.allclose(
    pc_cov,
    np.diag(eigenvalues)
)

print("PCA validation passed.")
print("Smallest eigenvalue:", eigenvalues.min())

In [ ]:
plt.figure(figsize=(9, 4))

plt.plot(
    range(1, n_stocks + 1),
    eigenvalues,
    marker="o"
)

plt.xlabel("Principal Component")
plt.ylabel("Eigenvalue")
plt.title("Residual Covariance PCA — Scree Plot")
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(9, 4))

plt.plot(
    range(1, n_stocks + 1),
    cumulative_variance,
    marker="o"
)

plt.axhline(
    0.80, linestyle="--", label="80%"
)

plt.axhline(
    0.90, linestyle="--", label="90%"
)

plt.xlabel("Number of Principal Components")
plt.ylabel("Cumulative Explained Variance")
plt.title("Cumulative Residual Variance Explained")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
df_residual_cov.to_excel(
    OUT / "equities_residual_cov_4w.xlsx",
    index_label="stock"
)

df_pca_summary.to_excel(
    OUT / "equities_residual_pca_summary.xlsx",
    index=False
)

df_eigenvectors.to_excel(
    OUT / "equities_residual_pca_eigenvectors.xlsx",
    index_label="stock"
)

df_pc_returns.to_excel(
    OUT / "equities_residual_pca_scores_4w.xlsx",
    index_label="date"
)